# 04 - Price Model Training

This notebook trains machine learning models to forecast electricity prices:
- Load feature-engineered dataset
- Train multiple model types (Linear, LightGBM, etc.)
- Evaluate model performance (MAE, RMSE, R²)
- Analyze feature importance
- Train quantile regression models for uncertainty estimation

## Outputs
- Trained price models
- Model evaluation metrics
- Feature importance analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.append('../')

from src.models import PriceModel, QuantilePriceModel, train_price_models
from src.utils import load_config, plot_feature_importance

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Features

In [ ]:
config = load_config('../configs/modelling_config.yaml')
data_path = Path(config.get('data.processed_path', '../data_processed'))
df_features = pd.read_csv(data_path / 'features.csv', index_col=0, parse_dates=True)

print(f"Features shape: {df_features.shape}")
print(f"Date range: {df_features.index.min()} to {df_features.index.max()}")

## 2. Prepare Training Data

In [ ]:
# Select features (exclude target and unnecessary columns)
exclude_cols = ['price', 'offshore_wind', 'onshore_wind', 'solar', 'demand']
feature_cols = [col for col in df_features.columns if col not in exclude_cols]

print(f"\nUsing {len(feature_cols)} features for modelling")
print(f"Sample features: {feature_cols[:10]}")

## 3. Train Price Models

In [ ]:
# Get model configuration
test_size = config.get('models.test_size', 0.2)
val_size = config.get('models.val_size', 0.1)
model_types = config.get('models.model_types', ['linear', 'lightgbm'])
quantiles = config.get('models.quantiles', [0.1, 0.5, 0.9])

# Train models
results = train_price_models(
    df_features,
    target_col='price',
    feature_cols=feature_cols,
    test_size=test_size,
    val_size=val_size,
    model_types=model_types,
    quantiles=quantiles
)

## 4. Evaluate Models

In [ ]:
# Display metrics
metrics_df = pd.DataFrame(results['metrics']).T
print("\nModel Performance Metrics:")
print(metrics_df)

# Plot metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, metric in enumerate(['mae', 'rmse', 'r2']):
    metrics_df[metric].plot(kind='bar', ax=axes[i], alpha=0.7)
    axes[i].set_title(f'{metric.upper()} Comparison')
    axes[i].grid(True, alpha=0.3, axis='y')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# Plot feature importance for best model
if 'lightgbm' in results['feature_importance']:
    importance = results['feature_importance']['lightgbm']
    plot_feature_importance(importance, top_n=20)

## 6. Predictions Visualization

In [ ]:
# Get test predictions
X_test = results['data_splits']['X_test']
y_test = results['data_splits']['y_test']

# Get best model
best_model_name = metrics_df['rmse'].idxmin()
best_model = results['models'][best_model_name]

if hasattr(best_model, 'predict'):
    y_pred = best_model.predict(X_test)
    
    # Plot actual vs predicted
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Time series
    axes[0].plot(y_test.index, y_test.values, label='Actual', alpha=0.7, linewidth=0.5)
    axes[0].plot(y_test.index, y_pred, label='Predicted', alpha=0.7, linewidth=0.5)
    axes[0].set_ylabel('Price (£/MWh)')
    axes[0].set_title(f'Actual vs Predicted - {best_model_name}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Scatter
    axes[1].scatter(y_test, y_pred, alpha=0.3, s=1)
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
    axes[1].set_xlabel('Actual Price (£/MWh)')
    axes[1].set_ylabel('Predicted Price (£/MWh)')
    axes[1].set_title('Prediction Scatter')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Quantile Predictions

In [ ]:
# Get quantile predictions
quantile_model = results['models']['quantile']
quantile_preds = quantile_model.predict(X_test)

# Plot quantile predictions
fig, ax = plt.subplots(figsize=(14, 6))
sample_range = slice(0, min(168, len(y_test)))  # First week

ax.plot(y_test.index[sample_range], y_test.iloc[sample_range], label='Actual', linewidth=2, color='black')
ax.plot(y_test.index[sample_range], quantile_preds['p50'].iloc[sample_range], label='P50', linewidth=2)
ax.fill_between(
    y_test.index[sample_range],
    quantile_preds['p10'].iloc[sample_range],
    quantile_preds['p90'].iloc[sample_range],
    alpha=0.3,
    label='P10-P90 Range'
)
ax.set_ylabel('Price (£/MWh)')
ax.set_title('Quantile Predictions (First Week of Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Models

In [ ]:
# Save models
if config.get('models.save_models', True):
    models_path = Path(config.get('models.model_save_path', '../models'))
    models_path.mkdir(parents=True, exist_ok=True)
    
    for name, model in results['models'].items():
        if name != 'quantile':
            filepath = models_path / f'{name}_model.pkl'
            model.save(str(filepath))
        else:
            filepath = models_path / 'quantile_models.pkl'
            model.save(str(filepath))
    
    print(f"\n✓ Models saved to {models_path}")

## Summary

Price modelling complete!

**Key results:**
- Multiple models trained and evaluated
- Feature importance analyzed
- Quantile predictions for uncertainty

**Next steps:**
- Proceed to notebook 05 for scenario modelling
- Use price models for forward projections
- Run Monte Carlo simulations